In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import classification_report

df = pd.read_csv('../data/transactions.csv', parse_dates=['date'])
print(f"Loaded {len(df)} transactions ✅")
print(f"Columns: {list(df.columns)}")
print(f"Known anomalies: {df['is_anomaly'].sum()}")

Loaded 1079 transactions ✅
Columns: ['date', 'category', 'amount', 'is_anomaly']
Known anomalies: 20


In [2]:
df['z_score'] = stats.zscore(df['amount'])
df['flag_zscore'] = (df['z_score'].abs() > 2.5).astype(int)
print(f"Flagged by Z-Score: {df['flag_zscore'].sum()}")

Flagged by Z-Score: 9


In [3]:
df['flag_iqr'] = 0

for cat in df['category'].unique():
    mask = df['category'] == cat
    subset = df.loc[mask, 'amount']
    Q1 = subset.quantile(0.25)
    Q3 = subset.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df.loc[mask, 'flag_iqr'] = ((subset < lower) | (subset > upper)).astype(int)

print(f"Flagged by IQR: {df['flag_iqr'].sum()}")

Flagged by IQR: 26


In [4]:
df['flag_combined'] = ((df['flag_zscore'] == 1) & 
                        (df['flag_iqr'] == 1)).astype(int)

print(f"Flagged by Z-Score : {df['flag_zscore'].sum()}")
print(f"Flagged by IQR     : {df['flag_iqr'].sum()}")
print(f"Flagged by BOTH    : {df['flag_combined'].sum()}")

Flagged by Z-Score : 9
Flagged by IQR     : 26
Flagged by BOTH    : 9


In [5]:
print("=" * 45)
print("METHOD 1 — Z-Score")
print("=" * 45)
print(classification_report(df['is_anomaly'], df['flag_zscore']))

print("=" * 45)
print("METHOD 2 — IQR")
print("=" * 45)
print(classification_report(df['is_anomaly'], df['flag_iqr']))

print("=" * 45)
print("METHOD 3 — Ensemble")
print("=" * 45)
print(classification_report(df['is_anomaly'], df['flag_combined']))

METHOD 1 — Z-Score
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1059
           1       1.00      0.45      0.62        20

    accuracy                           0.99      1079
   macro avg       0.99      0.72      0.81      1079
weighted avg       0.99      0.99      0.99      1079

METHOD 2 — IQR
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1059
           1       0.73      0.95      0.83        20

    accuracy                           0.99      1079
   macro avg       0.86      0.97      0.91      1079
weighted avg       0.99      0.99      0.99      1079

METHOD 3 — Ensemble
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1059
           1       1.00      0.45      0.62        20

    accuracy                           0.99      1079
   macro avg       0.99      0.72      0.81      1079
weighted avg       0

In [6]:
df.to_csv('../data/transactions_flagged.csv', index=False)
print("Saved ✅")

Saved ✅
